# ДЗ-2. Витрина клиента

Вы — аналитик ПРАЙМ, и у вас свой **сегмент клиентов**: его прислал бот 26 сентября, он же лежит в вашей строке `prime.assignments`, в колонке `client_filter`. Окно фактов у всех общее — первое полугодие 2026 года.

Задача недели — собрать **витрину клиента**: таблицу, в которой одна строка — один клиент, а рядом всё, что мы про него знаем. Из витрины потом считается всё остальное, поэтому лишние деньги в ней становятся лишними везде.

Идём по лестнице. Сначала вы дописываете один аргумент, как на семинаре, потом целую строку, потом пишете задачу сами, а в конце собираете витрину и доказываете протоколом, что двойного счёта нет.

### Как устроен ноутбук

| Часть | Что там | Что делаете вы |
|---|---|---|
| **1. Подготовка** | готовый код: подключение, ваш сегмент, выгрузка восьми таблиц | запускаете |
| **2. Уровень 1** | задачи 1–4: не хватает одного аргумента | вписываете аргумент вместо `ЗАПОЛНИТЕ` |
| **3. Уровень 2** | задачи 5–8: не хватает строки | дописываете строку |
| **4. Уровень 3** | задачи 9–12 | пишете код целиком |
| **5. Уровень 4** | задача 13: витрина и протокол сборки | пишете код целиком |
| **6. Итог** | готовый код: ваши ответы и проверка их формата | запускаете перед сдачей |
| **7. Исследование** | две части для тех, кто хочет 9 или 10 | по желанию |

После каждого уровня — вопрос, на который вы отвечаете словами в ячейке «✍️ Ответ на вопрос N».

**`ЗАПОЛНИТЕ`** — место, куда вписать ваш код. Пока оно не заполнено, ячейка падает с `NameError: name 'ЗАПОЛНИТЕ' is not defined` — так и задумано.

**Оценка.** Части 1–6 дают до 8 баллов: сделали всё аккуратно — 8, и это «отлично». 9 и 10 ставятся только за исследование в части 7. Все критерии и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-2-витрина-клиента).

Запускайте ячейки сверху вниз (`Shift + Enter`). Перед сдачей выполните **Kernel → Restart Kernel and Run All Cells**: ноутбук должен пройти целиком без красного.

Ноутбук лежит в вашем репозитории как `notebooks/hw2.ipynb`, JupyterLab запускается из корня `prime-monitor` командой `uv run jupyter lab`.

## Часть 1. Подготовка — готовый код

Код в этой части писать не нужно: запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Подключение | подключается к базе через ваш `.env` и заводит функцию `q(sql)` |
| 1.2 Ваш сегмент | достаёт условие `client_filter` из вашей строки `prime.assignments` |
| 1.3 Выгрузка | загружает восемь таблиц — дальше всё в pandas |
| 1.4 Формат ответов | заводит служебную функцию для итога |

Весь SQL задания собран здесь, в готовом коде. Дальше вы работаете только в pandas.

### 1.1 Подключение

Функция `q(sql)` отправляет запрос в базу и возвращает `DataFrame`, как на семинарах. Соединение открывается на каждый запрос и сразу закрывается.

Если здесь ошибка `Пакет prime не найден`, ноутбук запущен не в окружении вашего репозитория. Если `Не найдена переменная PRIME_DSN`, в корне `prime-monitor` нет файла `.env`.

In [1]:
import json
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Моменты времени считаем в UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

подключились как: kausmanova


### 1.2 Ваш сегмент

Сегмент — это клиенты, которые встречаются в вашей таблице с вашим признаком за ваш период. Бот прислал его вам 26 сентября готовым условием для `where`, оно же лежит в вашей строке `prime.assignments`.

**Эту ячейку не меняйте.** При проверке преподаватель подставит сюда условие **другого** сегмента, и все ответы должны пересчитаться сами. Поэтому ниже нигде не вписывайте руками ни условие, ни числа своего сегмента.

In [2]:
CLIENT_FILTER = q("select client_filter from prime.assignments where login = current_user").iloc[0, 0]

print(CLIENT_FILTER)

client_id in (
    select client_id
    from prime.service_usage
    where device_type = 'desktop'
      and started_at >= '2026-05-01 00:00+00'
      and started_at <  '2026-06-01 00:00+00'
)


### 1.3 Выгрузка

Восемь таблиц: первые пять — про ваш сегмент, последние три — справочник тарифов и данные всей базы.

| Переменная | Одна строка — это | Что внутри |
|---|---|---|
| `clients` | клиент сегмента | `client_id`, дата регистрации, регион, канал привлечения |
| `subs` | подписка, которой владеет клиент сегмента | тариф, даты начала и конца, пробная ли, причина отмены |
| `pay` | попытка списания по этим подпискам за окно фактов | сумма, **статус**, способ оплаты |
| `sub_members` | участник подписки из `subs` | `subscription_id` и `member_id` — кто подключён к подписке |
| `members` | участие клиента сегмента в любой подписке | `subscription_id` и `client_id` — клиент сегмента как участник |
| `tariffs` | цена тарифа в интервале дат | весь справочник, пять строк |
| `spend` | расходы канала за день | `channel`, `spend_date`, `amount` |
| `registrations` | канал × день регистрации | `n_clients` — сколько клиентов всей базы пришло из канала в этот день |

**Окно фактов общее для всех:** с 1 января по 30 июня 2026 года включительно. В запросе правый конец не включён: `paid_at < '2026-07-01'`.

Самые большие сегменты выгружаются около минуты и занимают в памяти до 2 ГБ — закройте лишние вкладки браузера. Если выгрузка идёт дольше двух минут, сервер оборвёт запрос — что делать, написано в тексте задания, раздел «Если что-то не работает».

In [3]:
WINDOW = ("2026-01-01", "2026-07-01")   # окно фактов: первое полугодие 2026, правый конец не включён
SEGMENT = f"select client_id from prime.clients where {CLIENT_FILTER}"

clients = q(f"""
    select client_id, registered_at, region_code, acquisition_channel
    from prime.clients
    where {CLIENT_FILTER}
    order by client_id
""")

subs = q(f"""
    select subscription_id, owner_client_id, tariff_id, started_at, ended_at, is_trial, cancel_reason
    from prime.subscriptions
    where owner_client_id in ({SEGMENT})
    order by subscription_id
""")

pay = q(f"""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount, p.status, p.payment_method
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where s.owner_client_id in ({SEGMENT})
      and p.paid_at >= :a and p.paid_at < :b
    order by p.payment_id
""", a=WINDOW[0], b=WINDOW[1])

sub_members = q(f"""
    select m.subscription_id, m.client_id as member_id
    from prime.subscription_members m
    join prime.subscriptions s on s.subscription_id = m.subscription_id
    where s.owner_client_id in ({SEGMENT})
    order by m.subscription_id, m.client_id
""")

members = q(f"""
    select subscription_id, client_id
    from prime.subscription_members
    where client_id in ({SEGMENT})
    order by client_id, subscription_id
""")

tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")

spend = q("select channel, spend_date, amount from prime.marketing_spend order by channel, spend_date")

registrations = q("""
    select acquisition_channel as channel, registered_at::date as reg_date, count(*) as n_clients
    from prime.clients
    group by 1, 2
    order by 1, 2
""")

N = len(clients)
print("ЗЕРНО витрины: одна строка — один клиент сегмента. Клиентов:", N)
print()
for name, frame in [("clients", clients), ("subs", subs), ("pay", pay), ("sub_members", sub_members),
                    ("members", members), ("tariffs", tariffs), ("spend", spend),
                    ("registrations", registrations)]:
    print(f"{name:<14} {len(frame):>9} строк")

ЗЕРНО витрины: одна строка — один клиент сегмента. Клиентов: 71275

clients            71275 строк
subs               56916 строк
pay               248478 строк
sub_members        95464 строк
members           117731 строк
tariffs                5 строк
spend               3606 строк
registrations       6762 строк


### 1.4 Формат ответов

Служебная функция: по ней итог в части 6 проверяет, что ответ записан в нужном формате — целое, число с копейками, список. **Верно ли посчитано, она не знает и не сообщает**: это проверяет преподаватель. Код ячейки свёрнут, просто запустите её.

In [4]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 2 знаков — рубли",
    2: "int",
    3: "int",
    4: "float до 2 знаков — рубли (0.0, если таких клиентов нет)",
    5: "float до 2 знаков — рубли",
    6: "float до 2 знаков — рубли",
    7: "int",
    8: "float до 1 знака — проценты",
    9: "int",
    10: "list [float до 2 знаков, int]",
    11: "list [float до 2 знаков, int]",
    12: "list из 3 int",
    13: "list из 3 int — то, что вернула print_protocol()",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def money(x) -> bool:   # float, у которого не больше двух знаков после запятой
        return type(x) is float and round(x, 2) == x

    def pct(x) -> bool:     # float, у которого не больше одного знака после запятой
        return type(x) is float and round(x, 1) == x

    def whole(x) -> bool:
        return type(x) is int

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: money, 2: whole, 3: whole, 4: money, 5: money, 6: money, 7: whole, 8: pct,
        9: whole,
        10: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and whole(a[1]),
        11: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and whole(a[1]),
        12: lambda a: type(a) is list and len(a) == 3 and all(whole(x) for x in a),
        13: lambda a: type(a) is list and len(a) == 3 and all(whole(x) for x in a),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

проверка формата готова


## Часть 2. Уровень 1 — дыра в один аргумент

Как на семинаре: код готов, не хватает одного аргумента. Впишите его вместо `ЗАПОЛНИТЕ` и запустите ячейку.

Последняя строка каждой задачи кладёт ответ в `answer_N`. Эти строки и имена `answer_N` не меняйте: по ним работа проверяется автоматически, в том числе на другом сегменте.

**Не получилась задача — не оставляйте падающий код.** Ячейка с ошибкой останавливает Run All. Замените всё содержимое ячейки одной строкой `answer_N = None`, где N — номер задачи: итог покажет «❌ не решена», а остальное проверится как обычно.

**Ноутбук не скажет, правильно ли вы посчитали.** Итог в части 6 проверяет только формат. Правильность проверяет преподаватель — на вашем сегменте и на контрольном.

**В `pay` лежат все попытки списания:** `success`, `failed` и `refunded`. Какие из них деньги, решаете вы — это задача 1.

### Задача 1. Контрольное число ДЕНЬГИ

Первое правило сборки витрины: контрольное число считают до сборки. Мы заранее знаем, сколько денег принёс сегмент за окно, — и потом витрина обязана дать ровно столько же.

**Что сделать**

Оставьте в `pay` только платежи, которые действительно принесли деньги, и посчитайте их сумму.

**Формат ответа:** `float` до двух знаков, рубли — например `98765432.1`.

**Подсказка:** Какие статусы бывают, покажет `pay["status"].value_counts()`.

In [5]:
# Какие попытки списания — это деньги, которые остались у компании?
success = pay[pay["status"] == "success"]
revenue = success["amount"].sum()

print("платежей с этим статусом:", len(success))
answer_1 = round(float(revenue), 2)
answer_1

платежей с этим статусом: 237104


73450246.0

### Задача 2. Подписки без платежа

Руководитель спрашивает, сколько подписок сегмента за полгода не принесли ни рубля. Чтобы ответить, к каждой подписке присоединяют число её успешных платежей и смотрят, у кого пусто.

**Что сделать**

Выберите вид соединения так, чтобы подписки без платежей **остались** в результате — с пропуском в `n_payments`.

**Формат ответа:** `int` — например `4321`.

**Подсказка:** Слайд 7 Л4: `how=` решает судьбу строк, которым не нашлась пара.

In [6]:
# Сколько успешных платежей пришло по каждой подписке за окно.
by_sub = (pay[pay["status"] == "success"]
          .groupby("subscription_id", as_index=False)
          .agg(n_payments=("payment_id", "size")))

# Подписка без платежей должна остаться в результате — с пропуском в n_payments.
subs_pay = subs.merge(by_sub, on="subscription_id", how="left", validate="1:1")

no_payment = subs_pay["n_payments"].isna().sum()
print("подписок:", len(subs), " строк после соединения:", len(subs_pay))
answer_2 = int(no_payment)
answer_2

подписок: 56916  строк после соединения: 56916


9146

### Задача 3. Кто платил

Сколько владельцев из сегмента заплатили за полгода хотя бы раз? Для этого к каждому платежу приписывают владельца подписки. Но прежде чем соединять, объявляют кратность: сколько платежей приходится на одну подписку и сколько подписок — на один платёж.

**Что сделать**

Впишите в `validate=` кратность связи «платёж → подписка».

**Формат ответа:** `int` — например `5432`.

**Подсказка:** Слайд 6 Л4, четыре случая кратности. Если слева на один ключ приходится много строк, это «m».

In [7]:
success = pay[pay["status"] == "success"]

# Сколько платежей приходится на одну подписку — и сколько подписок на один платёж?
pay_owner = success.merge(
    subs[["subscription_id", "owner_client_id"]],
    on="subscription_id", how="left", validate="m:1",
)

payers = pay_owner["owner_client_id"].nunique()
print("платежей до соединения:", len(success), " после:", len(pay_owner))
answer_3 = int(payers)
answer_3

платежей до соединения: 237104  после: 237104


47770

### Задача 4. Выручка без канала

Маркетинг смотрит выручку по каналам привлечения. У части клиентов канал не записан, и если этого не заметить, сумма по каналам тихо разойдётся с общей выручкой.

**Что сделать**

Сделайте так, чтобы клиенты без канала попали в отчёт отдельной строкой, и посчитайте их выручку.

**Формат ответа:** `float` до двух знаков, рубли — например `123456.7`. Если таких клиентов в сегменте нет — `0.0`.

**Подсказка:** Слайд 18 Л4: сумма по группам не сходится с общей. Сверьте сумму по каналам с выручкой задачи 1.

In [8]:
success = pay[pay["status"] == "success"]
pay_client = (success
              .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
              .merge(clients[["client_id", "acquisition_channel"]],
                     left_on="owner_client_id", right_on="client_id", validate="m:1"))

# Клиенты без канала должны остаться в отчёте отдельной строкой.
by_channel = pay_client.groupby("acquisition_channel", dropna=False)["amount"].sum()

print(by_channel.round(2).to_string())
no_channel = by_channel[by_channel.index.isna()].sum()
answer_4 = round(float(no_channel), 2)
answer_4

acquisition_channel
crm                 9420876.0
organic            21616469.0
partner_telecom    12591351.0
performance        13397105.0
referral            6541508.0
social              8387194.0
NaN                 1495743.0


1495743.0

**Вопрос 1.** В `pay` есть попытки списания со статусом `refunded`, и сумма у них положительная. Сколько таких строк в вашем сегменте и на какую сумму? Почему эту сумму нельзя прибавить к выручке — и почему её нельзя вычесть из выручки?

In [9]:
refunded = pay[pay["status"] == "refunded"]

n_refunded = len(refunded)
sum_refunded = round(float(refunded["amount"].sum()), 2)
share_rows = round(100 * n_refunded / len(pay), 2)

print("строк refunded:", n_refunded)
print("на сумму:", sum_refunded, "₽")
print("доля строк pay:", share_rows, "%")
# Числа для ответа на вопрос 1 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.

строк refunded: 3044
на сумму: 944056.0 ₽
доля строк pay: 1.23 %


#### ✍️ Ответ на вопрос 1
В моем сегменте 3044 попытки списания имеют статус refunded (это 1,23% строк pay на общую сумму 944056 рублей). Добавлять эту сумму к выручке нельзя, так как эти деньги были возвращены клиентам, поэтому фактической выручкой компании они не являются.
Вычитать 944056 из выручки тоже неправильно, потому что в расчет из задачи 1 уже вошли только платежи со статусом успешно (success), поэтому возвраты в сумму 73450246 рублей изначально не попали. Если их дополнительно вычесть, получится двойной учёт, и выручка составит 72506190 руб и будет занижена на 944056 рублей.

## Часть 3. Уровень 2 — дыра в строку

Теперь не хватает целой строки: выражения, условия или набора аргументов. Всё остальное готово. Правила те же: строки с `answer_N` не трогайте, не получилось — `answer_N = None`.

### Задача 5. Свернуть до клиента

Средний доход с платящего клиента — ARPPU — это выручка, делённая на число клиентов, которые платили. Чтобы посчитать его по витрине, платежи сначала сворачивают до зерна «клиент» и только потом соединяют с клиентами — иначе соединение перемножит строки.

**Что сделать**

Допишите `agg(...)`: для каждого владельца — `n_payments` (сколько успешных платежей) и `paid` (сколько он заплатил).

**Формат ответа:** `float` до двух знаков, рубли — например `1234.56`.

**Подсказка:** Слайд 17 Л4, именованные агрегаты: `agg(имя=("колонка", "функция"), ...)`.

In [10]:
success = pay[pay["status"] == "success"]

# Одна строка — один владелец: сколько успешных платежей и сколько денег.
by_client = (success
             .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
             .groupby("owner_client_id", as_index=False)
             .agg(n_payments=("payment_id", "size"), paid=("amount", "sum")))

mart5 = clients.merge(by_client, left_on="client_id", right_on="owner_client_id",
                      how="left", validate="1:1")

paying = mart5["n_payments"].notna().sum()
arppu = mart5["paid"].sum() / paying
print("клиентов:", len(mart5), " из них платили:", paying)
answer_5 = round(float(arppu), 2)
answer_5

клиентов: 71275  из них платили: 47770


1537.58

### Задача 6. Цена по справочнику

В `tariffs` лежит история цен: у тарифа несколько строк с интервалами действия, а одна строка ошибочная — вы нашли её на семинаре. Сколько принёс тариф «Расширенный» по цене справочника?

**Что сделать**

Допишите две строки: 1) справочник без ошибочной строки; 2) условие «момент платежа попадает в интервал действия цены» по конвенции курса.

**Формат ответа:** `float` до двух знаков, рубли — например `2345678.9`.

**Подсказка:** Раздел 2 семинара С4. Конвенция границы — `(paid_at >= valid_from) & (paid_at < valid_to + 1 день)`, потому что `valid_to` — это дата, то есть полночь. Каждый платёж должен найти в справочнике ровно одну цену: сравните число строк до и после.

In [11]:
ref = tariffs.copy() # Смотрю справочник тарифов, чтобы найти ошибочную строку
for col in ("valid_from", "valid_to"):
    ref[col] = pd.to_datetime(ref[col]).dt.tz_localize("UTC")
ref.sort_values(["tariff_id", "valid_from"])

,tariff_id,tariff_name,monthly_fee,cashback_rate,share_slots,valid_from,valid_to
0,1,Базовый,199.0,0.01,3,2024-01-01 00:00:00+00:00,2026-02-28 00:00:00+00:00
1,1,Базовый,249.0,0.01,3,2026-03-01 00:00:00+00:00,2030-12-31 00:00:00+00:00
2,2,Расширенный,399.0,0.03,5,2024-01-01 00:00:00+00:00,2026-02-28 00:00:00+00:00
3,2,Расширенный,429.0,0.03,5,2026-02-01 00:00:00+00:00,2026-06-30 00:00:00+00:00
4,2,Расширенный,449.0,0.03,5,2026-03-01 00:00:00+00:00,2030-12-31 00:00:00+00:00


In [12]:
# Даты справочника — дни без часового пояса, а платежи — моменты в UTC.
ref = tariffs.copy()
for col in ("valid_from", "valid_to"):
    ref[col] = pd.to_datetime(ref[col]).dt.tz_localize("UTC")

# 1. Справочник без строки, из-за которой интервалы одного тарифа перекрываются.
ref_fixed = ref.drop(index=3)

pay_t = (pay[pay["status"] == "success"]
         .merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1"))
m = pay_t.merge(ref_fixed[["tariff_id", "tariff_name", "monthly_fee", "valid_from", "valid_to"]],
                on="tariff_id")

# 2. Момент платежа попадает в интервал действия цены — по конвенции курса.
hit = (m["paid_at"] >= m["valid_from"]) & (m["paid_at"] < m["valid_to"] + pd.Timedelta(days=1))
priced = m[hit]

print("платежей:", len(pay_t), " строк с ценой:", len(priced))
extended = priced.loc[priced["tariff_name"] == "Расширенный", "monthly_fee"].sum()
answer_6 = round(float(extended), 2)
answer_6

платежей: 237104  строк с ценой: 237104


39050704.0

### Задача 7. Выше среднего по своему каналу

Маркетинг хочет знать, сколько платящих клиентов приносят больше, чем типичный клиент **их же** канала. Для этого рядом с каждым клиентом ставят среднее по его каналу — не сворачивая таблицу.

**Что сделать**

Допишите строку: `channel_avg` — среднее `paid` по каналу клиента, по значению в каждой строке.

**Формат ответа:** `int` — например `6543`.

**Подсказка:** Слайд 19 Л4: `transform` возвращает столько же строк, сколько было.

In [13]:
success = pay[pay["status"] == "success"]
per_client = (success
              .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
              .groupby("owner_client_id", as_index=False)
              .agg(paid=("amount", "sum"))
              .merge(clients[["client_id", "acquisition_channel"]],
                     left_on="owner_client_id", right_on="client_id", validate="1:1"))
per_client = per_client[per_client["acquisition_channel"].notna()]   # только известные каналы

# Рядом с каждым клиентом — среднее paid по его каналу. Строк остаётся столько же.
per_client["channel_avg"] = per_client.groupby("acquisition_channel")["paid"].transform("mean")

above = (per_client["paid"] > per_client["channel_avg"]).sum()
print("платящих клиентов с известным каналом:", len(per_client))
answer_7 = int(above)
answer_7

платящих клиентов с известным каналом: 46810


15285

### Задача 8. Выручка по месяцам

Финансам нужна таблица: строки — месяцы, колонки — тарифы, в клетках — выручка. По ней видно, как выручка «Базового» менялась от месяца к месяцу.

**Что сделать**

Допишите аргументы `pivot_table`: строки — `month`, колонки — `tariff_id`, значения — сумма `amount`.

**Формат ответа:** `float` до одного знака, проценты — например `87.6`.

**Подсказка:** Слайд 23 Л4.

In [14]:
success = (pay[pay["status"] == "success"]
           .merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1"))
success["month"] = success["paid_at"].dt.strftime("%Y-%m")

# Строки — месяцы, колонки — тарифы (1 — «Базовый», 2 — «Расширенный»), в клетках — выручка.
by_month = success.pivot_table(index="month", columns="tariff_id", values="amount", aggfunc="sum")

print(by_month.round(2).to_string())
april_to_march = by_month.loc["2026-04", 1] / by_month.loc["2026-03", 1] * 100
answer_8 = round(float(april_to_march), 1)
answer_8

tariff_id          1          2
month                          
2026-01    4566254.0  5554878.0
2026-02    4310539.0  5277972.0
2026-03    6281274.0  6911457.0
2026-04    5926947.0  6572462.0
2026-05    6760848.0  7518954.0
2026-06    6553680.0  7214981.0


94.4

**Вопрос 2.** Выручка «Базового» в апреле — `answer_8` процентов от мартовской. Это отток клиентов? Проверьте по дням: сколько дней апреля есть в `pay` и как выглядит выручка по дням. Объясните, что случилось, и что вы написали бы финансам в отчёте.

In [15]:
success = pay[pay["status"] == "success"].copy()
success["day"] = success["paid_at"].dt.strftime("%Y-%m-%d")
success["month"] = success["paid_at"].dt.strftime("%Y-%m")

# 1. Сколько разных дней с платежами в каждом месяце
days_per_month = success.groupby("month")["day"].nunique()
print(days_per_month.to_string())

# 2. Выручка по дням для марта и апреля
daily = (success[success["month"].isin(["2026-03", "2026-04"])]
         .groupby(["month", "day"])["amount"].sum())
print(daily.groupby("month").agg(["count", "mean", "min", "max"]).round(2).to_string())

# 3. Выручка в апреле по дням: нет ли дыры или аномального дня
print(daily.loc["2026-04"].round(2).to_string())
# Числа для ответа на вопрос 2 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.

month
2026-01    31
2026-02    28
2026-03    31
2026-04    28
2026-05    31
2026-06    30
         count       mean       min       max
month                                        
2026-03     31  425571.97  404450.0  458796.0
2026-04     28  446407.46  403499.0  488798.0
day
2026-04-01    452559.0
2026-04-02    434705.0
2026-04-03    403499.0
2026-04-04    466168.0
2026-04-05    413618.0
2026-04-06    444987.0
2026-04-07    426390.0
2026-04-08    466829.0
2026-04-09    428043.0
2026-04-10    436207.0
2026-04-11    457208.0
2026-04-12    446881.0
2026-04-13    438154.0
2026-04-16    437603.0
2026-04-17    470993.0
2026-04-18    460743.0
2026-04-19    443563.0
2026-04-20    445024.0
2026-04-21    455796.0
2026-04-22    453959.0
2026-04-23    436587.0
2026-04-24    451518.0
2026-04-25    445954.0
2026-04-26    459245.0
2026-04-27    436640.0
2026-04-28    449559.0
2026-04-29    448179.0
2026-04-30    488798.0


In [16]:
# Числа для ответа на вопрос 2 считайте здесь — по загруженным таблицам.
# сравниваю выручку Базового в марте и апреле в пересчёте на один день
base = (pay[pay["status"] == "success"]
        .merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1"))
base = base[base["tariff_id"] == 1].copy()
base["day"] = base["paid_at"].dt.strftime("%Y-%m-%d")
base["month"] = base["paid_at"].dt.strftime("%Y-%m")

per_day = (base[base["month"].isin(["2026-03", "2026-04"])]
           .groupby(["month", "day"])["amount"].sum())
stat = per_day.groupby("month").agg(days="count", total="sum", per_day="mean")
print(stat.round(0).to_string())
print("выручка в день, апрель к марту, %:",
      round(100 * stat.loc["2026-04", "per_day"] / stat.loc["2026-03", "per_day"], 1))

# Каких дат апреля нет совсем
all_days = pd.date_range("2026-04-01", "2026-04-30").strftime("%Y-%m-%d")
print("нет платежей:", sorted(set(all_days) - set(base.loc[base["month"] == "2026-04", "day"])))

         days      total   per_day
month                             
2026-03    31  6281274.0  202622.0
2026-04    28  5926947.0  211677.0
выручка в день, апрель к марту, %: 104.5
нет платежей: ['2026-04-14', '2026-04-15']


#### ✍️ Ответ на вопрос 2

Выручка базового в апреле составила 94,4% от мартовской выручки. Мне кажется, что это не отток клиентов, потому что в апрельских данных pay находятся платежи только за 28 дней из 30 (отсутствуют 14 и 15 апреля, а в марте 31 день).

Если сравнить выручку в расчете на один день с платежами, то в марте это 202 622 рублей в день, а в апреле - 211 677 рублей. То есть дневная выручка в апреле составила 104,5% от мартовской. Поэтому, по моему мнению, месячную выручку марта и апреля напрямую сравнивать неправильно. Для сравнения лучше использовать выручку в расчете на день с платежами, а по 14 и 15 апреля необходимо проверить полноту данных. 

## Часть 4. Уровень 3 — пишете сами

Заготовок больше нет: в ячейке только условие в комментарии и строка с `answer_N`. Код вы пишете целиком — от загруженных таблиц до ответа. Печатайте промежуточные результаты: так проще найти ошибку и объяснить решение на защите.

Каждая задача опирается на приём с лекции Л4 и семинара С4 — номер слайда стоит в подсказке. Ответ кладите в `answer_N` обычным `int` или `float`, а не `numpy`-числом: `int(...)`, `round(float(...), 2)`.

### Задача 9. Люди, а не места

Сколько людей сегмента пользуются ПРАЙМ? Пользуется тот, кто владеет подпиской или участвует хотя бы в одной чужой. Один человек может участвовать в нескольких подписках: если не свернуть участия до клиента, вы насчитаете места, а не людей.

**Что сделать**

Посчитайте клиентов сегмента, которые владеют подпиской (`subs`) или участвуют хотя бы в одной (`members`). Одна строка — один клиент.

**Формат ответа:** `int` — например `7654`.

**Подсказка:** Раздел 3 семинара С4: свернуть участия до клиента, соединить с `validate="1:1"`, флаг «владелец или участник».

In [17]:
# Задача 9. Сколько клиентов сегмента пользуются ПРАЙМ: владеют подпиской или участвуют хотя бы в одной. 
owners = (subs.groupby("owner_client_id", as_index=False)
              .agg(n_owned=("subscription_id", "size")))

# 2. Участники: одна строка на клиента сегмента, который участвует хотя бы в одной подписке
participants = (members.groupby("client_id", as_index=False)
                       .agg(n_member=("subscription_id", "size")))

# 3. К каждому клиенту сегмента приписываем оба признака. left сохраняет всех клиентов
mart9 = (clients[["client_id"]]
         .merge(owners, left_on="client_id", right_on="owner_client_id", how="left", validate="1:1")
         .merge(participants, on="client_id", how="left", validate="1:1"))

# 4. Флаги: есть ли пара. Пропуск после left-соединения означает «нет»
mart9["is_owner"] = mart9["n_owned"].notna()
mart9["is_member"] = mart9["n_member"].notna()
mart9["uses_prime"] = mart9["is_owner"] | mart9["is_member"]

a = int(mart9["is_owner"].sum())
b = int(mart9["is_member"].sum())
c = int((mart9["is_owner"] & mart9["is_member"]).sum())
print("клиентов в витрине:", len(mart9))
print("владельцев:", a, " участников:", b, " и то и другое:", c)
print("проверка a + b - c:", a + b - c)
print("не пользуются ни владением, ни участием:", int((~mart9["uses_prime"]).sum()))
answer_9 = int(mart9["uses_prime"].sum())
answer_9

клиентов в витрине: 71275
владельцев: 56916  участников: 61177  и то и другое: 46818
проверка a + b - c: 71275
не пользуются ни владением, ни участием: 0


71275

### Задача 10. Сколько человек на подписку

Подпиской пользуются владелец и те, кого он подключил. Подключить можно не больше `share_slots` человек — это поле справочника тарифов. Сколько в среднем человек приходится на подписку сегмента и сколько подписок заполнены до предела?

**Что сделать**

1. Для каждой подписки из `subs`: один владелец плюс число её строк в `sub_members`.
2. Среднее по всем подпискам сегмента — первая часть ответа.
3. Присоедините к подпискам `share_slots` из `tariffs`. Подписок, где людей ровно `share_slots` + 1, — вторая часть ответа.
4. Напечатайте, у скольких подписок людей больше, чем `share_slots` + 1. Это инвариант: число обязано быть нулём.

**Формат ответа:** `[среднее, заполненных до предела]` — например `[3.21, 987]`: `float` до двух знаков и `int`.

**Подсказка:** Слайды 8 и 14 Л4. У одного `tariff_id` в справочнике несколько строк, и `validate="m:1"` об этом скажет. Какая часть справочника на самом деле нужна для `share_slots`?

In [18]:
# Задача 10. Люди на подписках сегмента, считая владельца, и лимит share_slots
# 1. Сколько человек подключено к каждой подписке (без владельца).
n_conn = (sub_members.groupby("subscription_id", as_index=False)
                     .agg(n_connected=("member_id", "size")))

# 2. Приписываем число подключённых к каждой подписке. left сохраняет подписки без подключённых
subs10 = subs[["subscription_id", "tariff_id"]].merge(
    n_conn, on="subscription_id", how="left", validate="1:1")
subs10["n_connected"] = subs10["n_connected"].fillna(0).astype(int)
subs10["people"] = 1 + subs10["n_connected"]      # владелец + подключенные

# 3. share_slots из справочника. Сначала проверяем, что внутри тарифа он не меняется по датам, тогда можно оставить по одной строке на тариф
assert (tariffs.groupby("tariff_id")["share_slots"].nunique() == 1).all(), "share_slots меняется внутри тарифа"
slots = tariffs[["tariff_id", "share_slots"]].drop_duplicates("tariff_id")
subs10 = subs10.merge(slots, on="tariff_id", how="left", validate="m:1")

# 4. Заполнены до предела и инвариант.
full = subs10["people"] == subs10["share_slots"] + 1
over = subs10["people"] > subs10["share_slots"] + 1

print("подписок:", len(subs10), " без подключённых:", int((subs10["n_connected"] == 0).sum()))
print("подписок без share_slots:", int(subs10["share_slots"].isna().sum()))
print("среднее людей на подписку:", round(float(subs10["people"].mean()), 2))
print("заполнены до предела:", int(full.sum()))
print("больше предела (должно быть 0):", int(over.sum()))
answer_10 = [round(float(subs10["people"].mean()), 2), int(full.sum())]
answer_10

подписок: 56916  без подключённых: 16827
подписок без share_slots: 0
среднее людей на подписку: 2.68
заполнены до предела: 11276
больше предела (должно быть 0): 0


[2.68, 11276]

### Задача 11. Сколько стоил клиент

CAC — сколько стоило привести одного клиента. Расходы на привлечение известны по каналу и дню, а не по клиенту: присоединить их к витрине напрямую нельзя. Поэтому CAC считают на его собственном зерне — «канал × месяц» — и только потом приписывают клиентам.

**Что сделать**

1. CAC канала за месяц — это расходы канала за месяц (`spend`), делённые на число всех клиентов базы, которые пришли из этого канала в этом месяце (`registrations`). Берите только полные месяцы расходов — с 2025-02 по 2026-07.
2. Каждому клиенту сегмента припишите CAC его канала в месяц его регистрации.
3. Посчитайте средний CAC по клиентам сегмента, у которых он определён, и число клиентов, у которых он не определён: канал не записан или регистрация вне этих месяцев.

**Формат ответа:** `[средний CAC, клиентов без CAC]` — например `[432.1, 876]`: `float` до двух знаков и `int`.

**Подсказка:** Слайд 28 Л4: `marketing_spend` не присоединяется к витрине — у него другое зерно. Месяц из даты: `pd.to_datetime(...).dt.strftime("%Y-%m")`. Соединение по двум колонкам: `on=["channel", "month"]`, и кратность объявите.

In [19]:
# Задача 11. CAC по каналу и месяцу, затем средний CAC клиентов сегмента
# Ответ — [средний CAC (float до 2 знаков), клиентов без CAC (int)]
FIRST, LAST = "2025-02", "2026-07"      # полные месяцы расходов

# 1. Расходы канала по месяцам (из дневных строк), только полные месяцы
sp = spend.copy()
sp["month"] = pd.to_datetime(sp["spend_date"]).dt.strftime("%Y-%m")
sp = sp[(sp["month"] >= FIRST) & (sp["month"] <= LAST)]
spend_cm = sp.groupby(["channel", "month"], as_index=False).agg(spend=("amount", "sum"))

# 2. Новые клиенты ВСЕЙ базы по каналу и месяцу (строки без канала не нужны: у них нет расходов)
rg = registrations.dropna(subset=["channel"]).copy()
rg["month"] = pd.to_datetime(rg["reg_date"]).dt.strftime("%Y-%m")
reg_cm = rg.groupby(["channel", "month"], as_index=False).agg(n_new=("n_clients", "sum"))

# 3. CAC на зерне «канал × месяц».
cac = spend_cm.merge(reg_cm, on=["channel", "month"], how="inner", validate="1:1")
cac["cac"] = cac["spend"] / cac["n_new"]
print("канал×месяц: расходы", len(spend_cm), " регистрации", len(reg_cm), " после соединения", len(cac))

# 4. Каждому клиенту сегмента - CAC его канала в месяц его регистрации
cl = clients[["client_id", "registered_at", "acquisition_channel"]].copy()
cl["month"] = pd.to_datetime(cl["registered_at"]).dt.strftime("%Y-%m")
cl11 = cl.merge(cac[["channel", "month", "cac"]],
                left_on=["acquisition_channel", "month"], right_on=["channel", "month"],
                how="left", validate="m:1")

n_no_cac = int(cl11["cac"].isna().sum())
print("клиентов:", len(cl11), " с CAC:", int(cl11["cac"].notna().sum()), " без CAC:", n_no_cac)
print("из них без канала:", int(cl11["acquisition_channel"].isna().sum()))
print("средний CAC по клиентам:", round(float(cl11["cac"].mean()), 2))

answer_11 = [round(float(cl11["cac"].mean()), 2), n_no_cac]
answer_11

канал×месяц: расходы 108  регистрации 192  после соединения 108
клиентов: 71275  с CAC: 41610  без CAC: 29665
из них без канала: 1438
средний CAC по клиентам: 657.35


[657.35, 29665]

### Задача 12. Лучшие регионы канала

Где продавать ПРАЙМ дальше? Маркетинг просит три лучших региона по выручке в каждом канале привлечения — и начинает с самого доходного канала.

**Что сделать**

Посчитайте выручку сегмента за окно по каналу и региону владельца; клиенты без канала не участвуют. В каждом канале отберите три региона с наибольшей выручкой. Ответ — три `region_code` самого доходного канала сегмента, от большей выручки к меньшей.

**Формат ответа:** список из трёх `int` — например `[12, 34, 56]`.

**Подсказка:** Слайд 25 Л4: «три лучших вообще» и «три лучших в каждом канале» — разные вещи.

In [20]:
# Задача 12. Три лучших региона в каждом канале; ответ — для самого доходного канала.
# Ответ — список из трёх int, от большей выручки к меньшей.
# 1. Успешные платежи с владельцем подписки, его каналом и регионом
pay_reg = (pay[pay["status"] == "success"]
           .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
           .merge(clients[["client_id", "acquisition_channel", "region_code"]],
                  left_on="owner_client_id", right_on="client_id", validate="m:1"))
pay_reg = pay_reg[pay_reg["acquisition_channel"].notna()]    # клиенты без канала не участвуют
print("платежей с известным каналом:", len(pay_reg), " без региона:", int(pay_reg["region_code"].isna().sum()))

# 2. Выручка по каналу и региону
rev = (pay_reg.groupby(["acquisition_channel", "region_code"], as_index=False)
              .agg(revenue=("amount", "sum")))

# 3. Три лучших региона В КАЖДОМ канале (а не три лучших вообще)
rev = rev.sort_values(["acquisition_channel", "revenue"], ascending=[True, False])
top3 = rev.groupby("acquisition_channel").head(3)

# 4. Самый доходный канал сегмента и три его региона
by_channel = rev.groupby("acquisition_channel")["revenue"].sum().sort_values(ascending=False)
best_channel = by_channel.index[0]
print(by_channel.round(0).to_string())
print("самый доходный канал:", best_channel)
print(rev[rev["acquisition_channel"] == best_channel].head(5).to_string(index=False))
print(top3[top3["acquisition_channel"] == best_channel].to_string(index=False))

best = top3[top3["acquisition_channel"] == best_channel]

answer_12 = [int(r) for r in best["region_code"]]
answer_12

платежей с известным каналом: 232297  без региона: 0
acquisition_channel
organic            21616469.0
performance        13397105.0
partner_telecom    12591351.0
crm                 9420876.0
social              8387194.0
referral            6541508.0
самый доходный канал: organic
acquisition_channel  region_code  revenue
            organic           52 304365.0
            organic           21 284697.0
            organic           65 283706.0
            organic           28 281339.0
            organic           12 280398.0
acquisition_channel  region_code  revenue
            organic           52 304365.0
            organic           21 284697.0
            organic           65 283706.0


[52, 21, 65]

**Вопрос 3.** Куда вы добавили бы рубль на привлечение? Сравните каналы по двум числам: CAC канала (задача 11) и выручка на клиента вашего сегмента из этого канала. Назовите канал и оба числа. Почему выручку вашего сегмента нельзя просто разделить на CAC канала — какую оговорку нужно сделать?

In [21]:
# Числа для ответа на вопрос 3 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.
# 1. CAC канала за весь период: расходы / новые клиенты ВСЕЙ базы (таблица cac из задачи 11, полные месяцы 2025-02…2026-07). Суммы делим друг на друга, а не усредняем месячные CAC:так месяцы весят пропорционально числу привлечённых клиентов.
cac_ch = (cac.groupby("channel", as_index=False)
             .agg(spend=("spend", "sum"), n_new=("n_new", "sum"))
             .rename(columns={"channel": "acquisition_channel"}))
cac_ch["cac_channel"] = cac_ch["spend"] / cac_ch["n_new"]

# 2. Выручка сегмента по каналу владельца и число клиентов сегмента в канале
#    groupby по умолчанию отбрасывает клиентов без канала: в вопросе они не участвуют
pay_ch = (pay[pay["status"] == "success"]
          .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
          .merge(clients[["client_id", "acquisition_channel"]],
                 left_on="owner_client_id", right_on="client_id", validate="m:1"))
rev_ch = pay_ch.groupby("acquisition_channel", as_index=False).agg(revenue=("amount", "sum"))
n_ch = clients.groupby("acquisition_channel", as_index=False).agg(n_clients=("client_id", "size"))

# 3. Для сравнения: средний CAC по клиентам сегмента (как в задаче 11) и доля клиентов, у которых он определён
seg_cac = (cl11.groupby("acquisition_channel")
               .agg(cac_segment=("cac", "mean"), share_with_cac=("cac", lambda s: s.notna().mean()))
               .reset_index())

# 4. Сводка по каналам
q3 = (cac_ch
      .merge(rev_ch, on="acquisition_channel", how="left", validate="1:1")
      .merge(n_ch, on="acquisition_channel", how="left", validate="1:1")
      .merge(seg_cac, on="acquisition_channel", how="left", validate="1:1"))
q3["rev_per_client"] = q3["revenue"] / q3["n_clients"]
q3["rev_to_cac"] = q3["rev_per_client"] / q3["cac_channel"]

cols = ["acquisition_channel", "cac_channel", "cac_segment", "rev_per_client", "rev_to_cac", "share_with_cac"]
print(q3[cols].sort_values("rev_to_cac", ascending=False).round(2).to_string(index=False))

# Сверки: выручка и клиенты по каналам должны сходиться с общими числами минус клиенты без канала
print("выручка по каналам:", round(float(q3["revenue"].sum()), 2),
      " клиентов с каналом:", int(q3["n_clients"].sum()), "из", len(clients))
print("каналов без выручки или без CAC:", int(q3[["revenue", "cac_channel"]].isna().any(axis=1).sum()))

acquisition_channel  cac_channel  cac_segment  rev_per_client  rev_to_cac  share_with_cac
            organic       137.45       137.65         1166.44        8.49            0.57
                crm       213.18       213.01         1084.98        5.09            0.59
           referral       347.86       347.84         1113.07        3.20            0.58
    partner_telecom       674.83       674.02         1021.20        1.51            0.60
             social       812.39       812.54          838.38        1.03            0.62
        performance      1532.08      1535.73          929.64        0.61            0.61
выручка по каналам: 71954503.0  клиентов с каналом: 69837 из 71275
каналов без выручки или без CAC: 0


#### ✍️ Ответ на вопрос 3

Рубль на привлечение я бы добавила в канал organic, так как CAC здесь составляет 137,45 руб, а выручка на клиента моего сегмента за полгода - 1 166,44руб (это примерно в 8,5 раз больше CAC), и это самый лучший результат среди всех каналов. Например, у crm оно составляет 5,1, а у perfomance всего 0,61 (то есть выручка на клиента 929,64 не покрывает CAC в 1 532,08 ₽). 
Но напрямую сравнивать выручку сегмента с CAC канала некорректно, потому что CAC рассчитан по всем клиентам канала во всей базе, а выручка рассчитана только по моему сегменту (desktop в мае), который вполне может быть активнее среднего клиента канала.
Кроме того, 1166,44 рублей - это выручка за полгода, а не прибыль, поэтому показатель 8,5 хорошо подходит для сравнения каналов между собой, но использовать его как прогноз окупаемости не совсем правильно. 

## Часть 5. Уровень 4 — витрина и протокол сборки

Главная задача недели. Вы собираете **витрину клиента**: одна строка — один клиент сегмента, рядом всё, что про него известно. И доказываете, что двойного счёта нет, **протоколом сборки** — той же формы, что на слайде 15 Л4 и на семинаре С4.

> **Протокол — это 3 балла из 10.** Без него ноль, даже если все числа верные: результат, который нельзя перепроверить, в работе не принимают.

Три последние строки протокола — инварианты витрины:

- **ЗЕРНО:** строк в витрине столько же, сколько уникальных клиентов, и столько, сколько вы объявили до сборки;
- **ДЕНЬГИ:** сумма по витрине сходится с независимым контролем до копейки;
- **ЛЮДИ:** число тех, кто пользуется ПРАЙМ, сходится со счётом, сделанным другим путём.

Ниже готовые функции, эту ячейку просто запустите. `step()` — та же, что на семинаре: одна строка протокола на одно соединение. `check_step()` делает строку протокола для проверки из вашего `src/prime/checks.py`. `print_protocol()` печатает протокол и возвращает ответ задачи 13.

In [22]:
def step(name: str, before: pd.DataFrame, after: pd.DataFrame, key: str,
         validate: str = "", expect_lost: int | None = None) -> dict:
    """Одна строка протокола сборки.

    keys_lost — строки слева, которым не нашлось пары справа.
    Видно их по пропуску в колонке key, пришедшей из правой таблицы.
    """
    lost = int(after[key].isna().sum())
    if len(after) != len(before):
        verdict = f"ВНИМАНИЕ: строк было {len(before)}, стало {len(after)}"
    elif expect_lost is None:
        verdict = "ok"
    elif lost != expect_lost:
        verdict = f"ВНИМАНИЕ: потеряно {lost}, ждали {expect_lost}"
    else:
        verdict = f"ok, ожидали {expect_lost}"
    return {"step": name, "rows_before": len(before), "rows_after": len(after),
            "validate": validate, "keys_lost": lost, "verdict": verdict}


def check_step(name: str, frame: pd.DataFrame, bad: pd.DataFrame, expected: int) -> dict:
    """Строка протокола для проверки из checks.py.

    bad — строки, которые нашла проверка; expected — сколько вы ждали, объявив это ДО сборки.
    """
    found = len(bad)
    verdict = f"ok, ожидали {expected}" if found == expected else f"ВНИМАНИЕ: нашлось {found}, ждали {expected}"
    return {"step": name, "rows_before": len(frame), "rows_after": len(frame),
            "validate": "проверка", "keys_lost": found, "verdict": verdict}


def print_protocol(protocol: list[dict], mart: pd.DataFrame, declared: int,
                   money_mart: float, money_control: float,
                   people_mart: int, people_control: int) -> list[int]:
    """Напечатать протокол сборки. Вернуть [строк в витрине, разница ДЕНЬГИ в копейках, ЛЮДИ по витрине]."""
    rows, unique = len(mart), int(mart["client_id"].nunique())
    kopecks_mart, kopecks_control = round(money_mart * 100), round(money_control * 100)
    diff = kopecks_mart - kopecks_control

    print("ПРОТОКОЛ СБОРКИ ВИТРИНЫ: одна строка — один клиент сегмента")
    print()
    print(pd.DataFrame(protocol).to_string(index=False))
    print()
    print(f"ЗЕРНО   строк {rows}, клиентов {unique}, объявляли {declared}"
          f"   -> {'ok' if rows == unique == declared else 'ВНИМАНИЕ'}")
    print(f"ДЕНЬГИ  по витрине {money_mart:.2f} ₽, независимо {money_control:.2f} ₽, разница {diff} коп."
          f"   -> {'ok' if diff == 0 else 'ВНИМАНИЕ'}")
    print(f"ЛЮДИ    по витрине {people_mart}, независимо {people_control}"
          f"   -> {'ok' if people_mart == people_control else 'ВНИМАНИЕ'}")
    print()
    print("HW2_PROTOCOL=" + json.dumps({
        "steps": protocol, "grain": [rows, unique, declared],
        "money_kopecks": [kopecks_mart, kopecks_control], "people": [int(people_mart), int(people_control)],
    }, ensure_ascii=False, default=str))
    return [int(rows), int(diff), int(people_mart)]


print("функции протокола готовы")

функции протокола готовы


### Задача 13. Витрина клиента

Соберите витрину по своему сегменту и докажите протоколом, что двойного счёта нет. Весь раздел 4 семинара С4 — это тот же путь на общем сегменте.

**Что сделать**

1. **До сборки** объявите, сколько строк будет в витрине, и сколько ключей потеряет каждое левое соединение — эти числа пойдут в `expect_lost`.
2. Соберите `mart`, одна строка — один клиент сегмента:
   - `client_id`, `acquisition_channel`, `region_code` — из `clients`;
   - `subscription_id`, `tariff_id` — подписка клиента, пропуск, если её нет;
   - `n_payments`, `paid` — успешные платежи за окно, свёрнутые до клиента; `0`, если их нет;
   - `n_memberships` — в скольких подписках клиент участник; `0`, если ни в одной;
   - `uses_prime` — владелец или участник.
3. Каждое соединение запишите в протокол через `step()`.
4. Примените к витрине функцию из своего `src/prime/checks.py` — например, `find_missing_channel` — и запишите её строкой `check_step()` с ожиданием, объявленным до сборки. Это отдельные 0,5 балла из 10.
5. Посчитайте независимые контроли для ДЕНЬГИ и ЛЮДИ **не по витрине, а другим путём** и вызовите `print_protocol()`.

**Формат ответа:** то, что вернула `print_protocol()`: список из трёх `int`.

**Подсказка:** Если `src/prime/checks.py` у вас нет — создайте его по образцу семинара С3 и закоммитьте в `main`: `from prime.checks import find_missing_channel`. Протокол должен уметь покраснеть: проверьте его, сломав одно соединение, как на семинаре.

In [29]:
# Задача 13. Витрина клиента и протокол сборки.
# Одна строка — один клиент сегмента. Каждое соединение — через step()
# с ожиданием, объявленным ДО сборки. Проверка из checks.py — через check_step().
# В конце — answer_13 = print_protocol(...).

from prime.checks import find_missing_channel

protocol: list[dict] = []

# ДО СБОРКИ: Считаем множествами не через merge 
all_clients = set(clients["client_id"])
declared = len(all_clients)                                    # сколько строк будет в витрине

owner_of = subs.set_index("subscription_id")["owner_client_id"]
ok_pay = pay[pay["status"] == "success"]                       # деньги: только success, как в задаче 1
paying_owners = set(ok_pay["subscription_id"].map(owner_of))

exp_no_sub = len(all_clients - set(subs["owner_client_id"]))   # без подписки
exp_no_pay = len(all_clients - paying_owners)                  # без успешных платежей
exp_no_member = len(all_clients - set(members["client_id"]))   # не участвуют ни в одной подписке
exp_no_channel = int(clients["acquisition_channel"].isna().sum())   # ожидание для проверки из checks.py
print("объявлено: строк", declared, "| без подписки", exp_no_sub, "| без платежей", exp_no_pay,
      "| без участий", exp_no_member, "| без канала", exp_no_channel)

# Сборка: от clients только left

# Подписка клиента-владельца (у владельца одна подписка)
subs_by_client = subs[["owner_client_id", "subscription_id", "tariff_id"]].rename(
    columns={"owner_client_id": "client_id"})
m1 = clients[["client_id", "acquisition_channel", "region_code"]].merge(
    subs_by_client, on="client_id", how="left", validate="1:1")
protocol.append(step("clients <- subs: подписка", clients, m1, "subscription_id", "1:1", exp_no_sub))

# 2b. Успешные платежи, свернутые до клиента
pay_by_client = (ok_pay
                 .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
                 .groupby("owner_client_id", as_index=False)
                 .agg(n_payments=("payment_id", "size"), paid=("amount", "sum"))
                 .rename(columns={"owner_client_id": "client_id"}))
m2 = m1.merge(pay_by_client, on="client_id", how="left", validate="1:1")
protocol.append(step("mart <- платежи по клиенту", m1, m2, "n_payments", "1:1", exp_no_pay))

# Мембершипс, свернутые до клиента
members_by_client = members.groupby("client_id", as_index=False).agg(n_memberships=("subscription_id", "size"))
m3 = m2.merge(members_by_client, on="client_id", how="left", validate="1:1")
protocol.append(step("mart <- участия по клиенту", m2, m3, "n_memberships", "1:1", exp_no_member))

# Нули вместо пропусков и флажок владелец или участник
mart = m3.copy()
mart["n_payments"] = mart["n_payments"].fillna(0).astype(int)
mart["paid"] = mart["paid"].fillna(0.0)
mart["n_memberships"] = mart["n_memberships"].fillna(0).astype(int)
mart["uses_prime"] = mart["subscription_id"].notna() | (mart["n_memberships"] > 0)

# Проверка из checks.py к витрине
protocol.append(check_step("проверка: find_missing_channel(mart)", mart,
                           find_missing_channel(mart), exp_no_channel))

# Независимые контроли не по витрине  
money_mart = round(float(mart["paid"].sum()), 2)
money_control = round(float(ok_pay["amount"].sum()), 2)
people_mart = int(mart["uses_prime"].sum())
people_control = len((set(subs["owner_client_id"]) | set(members["client_id"])) & all_clients)

answer_13 = print_protocol(protocol, mart, declared,
                           money_mart, money_control, people_mart, people_control)
answer_13

объявлено: строк 71275 | без подписки 14359 | без платежей 23505 | без участий 10098 | без канала 1438
ПРОТОКОЛ СБОРКИ ВИТРИНЫ: одна строка — один клиент сегмента

                                step  rows_before  rows_after validate  keys_lost           verdict
           clients <- subs: подписка        71275       71275      1:1      14359 ok, ожидали 14359
          mart <- платежи по клиенту        71275       71275      1:1      23505 ok, ожидали 23505
          mart <- участия по клиенту        71275       71275      1:1      10098 ok, ожидали 10098
проверка: find_missing_channel(mart)        71275       71275 проверка       1438  ok, ожидали 1438

ЗЕРНО   строк 71275, клиентов 71275, объявляли 71275   -> ok
ДЕНЬГИ  по витрине 73450246.00 ₽, независимо 73450246.00 ₽, разница 0 коп.   -> ok
ЛЮДИ    по витрине 71275, независимо 71275   -> ok

HW2_PROTOCOL={"steps": [{"step": "clients <- subs: подписка", "rows_before": 71275, "rows_after": 71275, "validate": "1:1", "keys_lost": 14

[71275, 0, 71275]

In [30]:
# Проверка, что протокол краснеет:
# ломаем соединение участий — забываем свернуть members до клиента
# Ломаем намеренно: забываем свернуть участия и соединяем как есть.
broken = clients.merge(members, on="client_id", how="left")
seats = int(broken["subscription_id"].notna().sum())

print(f"строк: {len(broken)}, а клиентов {N}")
print(f"ЗЕРНО   -> {'ok' if len(broken) == N else 'ВНИМАНИЕ: одна строка больше не один клиент'}")
print(f"ЛЮДИ    -> такая таблица даёт {seats} МЕСТ вместо {people_control} ЛЮДЕЙ")

строк: 127829, а клиентов 71275
ЗЕРНО   -> ВНИМАНИЕ: одна строка больше не один клиент
ЛЮДИ    -> такая таблица даёт 117731 МЕСТ вместо 71275 ЛЮДЕЙ


**Вопрос 4.** Где ваш протокол заранее ждал потерь ключей и почему это не ошибка — назовите числа. Что стало бы со строками ЗЕРНО и ЛЮДИ, если бы участия не свернули до клиента? Посчитайте, а не угадывайте.

In [31]:
# Числа для ответа на вопрос 4 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.

# Где протокол ждал потерь ключей: берем из протокола и сверяем с объявленными ожиданиями
for row in protocol[:3]:
    print(f"{row['step']:<32} потеряно {row['keys_lost']:>6}  {row['verdict']}")

# Проверяю не пересекаются ли эти группы клиентов, Клиентов без подписки И без участий:
no_sub = all_clients - set(subs["owner_client_id"])
no_member = all_clients - set(members["client_id"])
print("без подписки:", len(no_sub), " без участий:", len(no_member),
      " и то и другое:", len(no_sub & no_member))

# Что было бы, если бы участия не свернули до клиента
broken = clients[["client_id"]].merge(members, on="client_id", how="left")
seats = int(broken["subscription_id"].notna().sum())
print("ЗЕРНО: строк", len(broken), "вместо", declared, "-> лишних", len(broken) - declared)
print("ЛЮДИ: мест", seats, "вместо людей", people_control, "-> лишних", seats - people_control)

clients <- subs: подписка        потеряно  14359  ok, ожидали 14359
mart <- платежи по клиенту       потеряно  23505  ok, ожидали 23505
mart <- участия по клиенту       потеряно  10098  ok, ожидали 10098
без подписки: 14359  без участий: 10098  и то и другое: 0
ЗЕРНО: строк 127829 вместо 71275 -> лишних 56554
ЛЮДИ: мест 117731 вместо людей 71275 -> лишних 46456


#### ✍️ Ответ на вопрос 4

Мой протокол заранее ждал потерь ключей в трех частях - 14359 клиентов без подписки, 23505 без успешных платежей и 10098 без участия в шеринговой подписке. До сборки я проверила эти значения через множества, и после соединений получилиcь те же числа. Это не является ошибочным, так как клиент без подписки не является владельцем, но участвует в чужой подписке, а без участий остаются те, кто только владеет ей (а пересечение этих групп равно нулю, то есть все 71275 клиентов пользуются праймом), а у 23505 клиентов нет успешных платежей за этот период. Если бы участия не свернули до клиента, было бы 127829 строк вместо 71275 и ЗЕРНО покраснело бы на 56554 лишних строки, а ЛЮДЕЙ насчитали бы 117731 место вместо 71275 человек (на 46 456 больше).

## Часть 6. Итог — готовый код

При **Restart Kernel and Run All Cells** эта ячейка выполняется после всех задач: перед сдачей посмотрите её вывод. Она показывает ваши ответы и проверяет только формат, но не правильность.

Последняя строка вывода служебная: по ней преподаватель сверяет ответы автоматически. Ячейку не удаляйте.

In [26]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 14)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте ответы словами — ячейки «✍️ Ответ на вопрос N» после каждого уровня.")
print("\n--- служебная строка, не удаляйте ---")
print("HW2_ANSWERS=" + json.dumps(
    {"client_filter": CLIENT_FILTER, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

задача  1: 73450246.0
    ✅ формат в порядке
задача  2: 9146
    ✅ формат в порядке
задача  3: 47770
    ✅ формат в порядке
задача  4: 1495743.0
    ✅ формат в порядке
задача  5: 1537.58
    ✅ формат в порядке
задача  6: 39050704.0
    ✅ формат в порядке
задача  7: 15285
    ✅ формат в порядке
задача  8: 94.4
    ✅ формат в порядке
задача  9: 71275
    ✅ формат в порядке
задача 10: [2.68, 11276]
    ✅ формат в порядке
задача 11: [657.35, 29665]
    ✅ формат в порядке
задача 12: [52, 21, 65]
    ✅ формат в порядке
задача 13: [71275, 0, 71275]
    ✅ формат в порядке

Не забудьте ответы словами — ячейки «✍️ Ответ на вопрос N» после каждого уровня.

--- служебная строка, не удаляйте ---
HW2_ANSWERS={"client_filter": "client_id in (\n    select client_id\n    from prime.service_usage\n    where device_type = 'desktop'\n      and started_at >= '2026-05-01 00:00+00'\n      and started_at <  '2026-06-01 00:00+00'\n)", "answers": {"1": 73450246.0, "2": 9146, "3": 47770, "4": 1495743.0, "5": 153

## Часть 7. Исследование — для тех, кто хочет 9 или 10

**Эта часть необязательна.** Части 1–6 — это оценка до 8 баллов, и 8 — это «отлично». 9 и 10 ставятся только за исследование: в нём две части, каждая до 1 балла сверху. Они засчитываются, если основная часть набрала не меньше 15 баллов из 20.

Здесь нет готового формата ответа и заготовок. Вы сами решаете, что посчитать, считаете это кодом и пишете вывод. Ассистент может помочь с кодом, но ответ зависит от того, что вы увидите в своих данных. Текст вывода — ваш: правило про LLM действует и здесь.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке, а не вписаны руками | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется у преподавателя целиком — в том числе на контрольном сегменте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Загруженных таблиц для исследования хватает: `pay`, `subs` и `tariffs`. Считайте от них, а не от чисел, вписанных руками, — на контрольном сегменте код должен отработать без правок.

### Исследование А. Что подорожание дало выручке

С 1 марта 2026 года «Базовый» подорожал с 199 до 249 ₽, «Расширенный» — с 399 до 449 ₽. Финансы сравнили выручку сегмента за март–июнь с январём–февралём и записали весь рост в заслугу подорожания. Руководитель просит проверить, сколько рублей в месяц на самом деле дала цена.

**Что сделать**

1. По месяцам окна посчитайте выручку и число успешных платежей — по каждому тарифу.
2. Разделите изменение выручки после 1 марта на две части. **Цена** — разница между тем, сколько стоили платежи марта–июня, и тем, сколько те же платежи стоили бы по старой цене. **Объём** — всё остальное.
3. Учтите две вещи, которые меняют число платежей в месяц без всякого подорожания. Подписка списывается раз в 30 дней, а месяцы разной длины. И в данных апреля есть дыра — вы видели её в вопросе 2.
4. Ответьте: сколько рублей в месяц приносит подорожание вашему сегменту — и насколько ошибается наивное сравнение «март–июнь против января–февраля».

In [32]:
# Исследование А: ваш код. Всё — от загруженных таблиц.
# Исследование А. Что подорожание дало выручке.
CUT = pd.Timestamp("2026-03-01", tz="UTC")     # день подорожания
MONTH_DAYS = 181 / 6                           # средний месяц окна: 181 день / 6 месяцев

# Цены до и после 1 марта: справочник без ошибочной строки (задача 6)
ref_r = tariffs.copy()
for col in ("valid_from", "valid_to"):
    ref_r[col] = pd.to_datetime(ref_r[col]).dt.tz_localize("UTC")
ref_r = ref_r.drop(index=3)
old_price = ref_r[ref_r["valid_from"] < CUT].set_index("tariff_id")["monthly_fee"]
new_price = ref_r[ref_r["valid_from"] >= CUT].set_index("tariff_id")["monthly_fee"]
print("цена до 1 марта:", old_price.to_dict(), " после:", new_price.to_dict())

# Успешные платежи с тарифом; по месяцам и тарифам — выручка, платежи, всё в день
ok_r = (pay[pay["status"] == "success"]
        .merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1"))
ok_r["day"] = ok_r["paid_at"].dt.strftime("%Y-%m-%d")
ok_r["month"] = ok_r["paid_at"].dt.strftime("%Y-%m")
ok_r["after"] = ok_r["paid_at"] >= CUT

n_days = ok_r.groupby("month")["day"].nunique().rename("days")
tab = (ok_r.groupby(["month", "tariff_id"])
           .agg(n=("payment_id", "size"), revenue=("amount", "sum"))
           .reset_index()
           .merge(n_days, on="month", validate="m:1"))
tab["n_per_day"] = (tab["n"] / tab["days"]).round(1)
tab["rev_per_day"] = (tab["revenue"] / tab["days"]).round(0)
print(tab.to_string(index=False))

# По периодам «до» и «после»: платежи и выручка, в расчёте на день с платежами
days = ok_r.groupby("after")["day"].nunique()
d_before, d_after = int(days.loc[False]), int(days.loc[True])
print("дней с платежами: до", d_before, " после", d_after)

g = (ok_r.groupby(["tariff_id", "after"])
         .agg(n=("payment_id", "size"), revenue=("amount", "sum"))
         .reset_index())
aft = g[g["after"]].set_index("tariff_id")
bef = g[~g["after"]].set_index("tariff_id")

# 3. Разложение. Цена = выручка после подорожания минус те же платежи по старой цене
price_total = float((aft["revenue"] - aft["n"] * old_price.reindex(aft.index)).sum())
price_day = price_total / d_after
delta_day = aft["revenue"].sum() / d_after - bef["revenue"].sum() / d_before
volume_day = delta_day - price_day

# 4. Наивное сравнение финансов: средняя месячная выручка март–июнь против января–февраля
by_month = ok_r.groupby("month")["amount"].sum()
naive = by_month[by_month.index >= "2026-03"].mean() - by_month[by_month.index < "2026-03"].mean()

res = pd.Series({
    "наивное сравнение, ₽/мес": naive,
    "изменение выручки на средний месяц, ₽/мес": delta_day * MONTH_DAYS,
    "из них цена, ₽/мес": price_day * MONTH_DAYS,
    "из них объём и всё остальное, ₽/мес": volume_day * MONTH_DAYS,
})
print(res.round(0).to_string())
print("наивное больше эффекта цены в", round(naive / (price_day * MONTH_DAYS), 2), "раза")

# 5. Чувствительность: если дыру 14–15 апреля считать днями без платежей, а не пропуском данных
cal_after = (pd.Timestamp("2026-07-01") - pd.Timestamp("2026-03-01")).days
price_cal = price_total / cal_after * MONTH_DAYS
print("цена, ₽/мес: по дням с платежами", round(price_day * MONTH_DAYS),
      " по календарным дням", round(price_cal))

цена до 1 марта: {1: 199.0, 2: 399.0}  после: {1: 249.0, 2: 449.0}
  month  tariff_id     n   revenue  days  n_per_day  rev_per_day
2026-01          1 22946 4566254.0    31      740.2     147299.0
2026-01          2 13922 5554878.0    31      449.1     179190.0
2026-02          1 21661 4310539.0    28      773.6     153948.0
2026-02          2 13228 5277972.0    28      472.4     188499.0
2026-03          1 25226 6281274.0    31      813.7     202622.0
2026-03          2 15393 6911457.0    31      496.5     222950.0
2026-04          1 23803 5926947.0    28      850.1     211677.0
2026-04          2 14638 6572462.0    28      522.8     234731.0
2026-05          1 27152 6760848.0    31      875.9     218092.0
2026-05          2 16746 7518954.0    31      540.2     242547.0
2026-06          1 26320 6553680.0    30      877.3     218456.0
2026-06          2 16069 7214981.0    30      535.6     240499.0
дней с платежами: до 59  после 120
наивное сравнение, ₽/мес                     3580329.

#### ✍️ Исследование А

Вопрос исследования - посчитать сколько выручки действительно добавило само повышение цены на тарифы. Для этого посчитала по каждому тарифу выручку и число успешных (!) платежей по месяцам, а потом привела их к одному дню с платежами, так как месяцы разной длительности, и кроме того в апреле в данных нет 14 и 15 числа. Затем разделила рост выручки на вклад цены в рост выручки (вклад цены в рост выручки посчитала как разницу между новой и старой ценой при том же количестве платежей) и объем. До повышения получилось 59 дней с платежами и 120 после. Итог: в среднем за месяц выручка выросла на 3,43 млн. рублей (при этом из них 2,08 млн дала сама цена, ещё 1,35 млн - это рост числа платежей и другие эффекты). При простом же сравнении получается 3,58 млн, поэтому такое сравнение завышает показатель примерно в 1.72 раза. Так как в предыдущих заданиях была выявлена дыра в 14-15 апреля, я пересчитала цену по календарным дням (получилось 2,04 млн/мес, то есть результат меняется примерно на 1,6%). То есть для моего сегмента повышение цены дало около 2,1 млн доп. выручки в месяц(но не 3,6 млн). Также хочу отметить, что  число платежей/день росло ещё до 1 марта: у «Базового» с 740 в январе до 774 в феврале, у «Расширенного» с 449 до 472. После марта рост тоже продолжился, но по моим данным нельзя точно сказать, каким он был бы без повышения. 

Вывод: 2,08 млн руб — это оценка эффекта самой цены (то есть вклад цены в рост выручки), если считать, что она не повлияла на количество платежей. Если из-за подорожания часть клиентов перестала платить, реальный эффект, конечно, будет ниже. Также хотела бы особенно выделить ограничения моего исследования: тренд до подорожания виден только по двум месяцам, поэтому его продолжение это предположение; дни 14 и 15 апреля я считала пропуском в данных, а не днями без платежей; все числа относятся только к моему сегменту

### Исследование Б. Стоило ли подорожание подписчиков

Маркетинг опасается, что подорожание вызвало волну отмен, и предлагает вернуть старые цены. Проверьте на своём сегменте, есть ли для этого основания.

**Что сделать**

1. По месяцам с января по июнь посчитайте отток: сколько подписок закончилось в месяце из тех, что были активны на его начало. Данные — `subs`, колонки `started_at` и `ended_at`.
2. Сравните отток до и после 1 марта — отдельно для «Базового» (+25 %) и «Расширенного» (+12,5 %). Если бы цена отпугивала, какой тариф должен был отреагировать сильнее?
3. Посмотрите на причины отмен: как менялась доля `too_expensive` среди отмен по месяцам.
4. Вывод для маркетинга: есть ли в данных след подорожания в оттоке — и чего ваш расчёт не доказывает. Сколько у вас отмен в месяц и хватит ли этого, чтобы заметить эффект?

In [33]:
# Исследование Б: ваш код. Всё — от загруженных таблиц.
# Исследование Б. Стоило ли подорожание подписчиков.

CUT = pd.Timestamp("2026-03-01", tz="UTC")  # день подорожания; tz нужен, чтобы сравнивать с моментами в UTC

s = subs.copy()
for c in ("started_at", "ended_at"):
    s[c] = pd.to_datetime(s[c], utc=True)

# Что лежит в данных (проверяю допущения)
#    (есть ли значение too_expensive) и сколько подписок вообще закончилось
print("подписок:", len(s), " с датой окончания:", int(s["ended_at"].notna().sum()))
print(s["cancel_reason"].value_counts(dropna=False).to_string())
print("пробных:", int(s["is_trial"].sum()))

# Отток по месяцам и тарифам: выбираю из активных на начало месяца сколько закончилось в месяце
#    Отток месяца = (сколько подписок, активных на начало месяца, закончилось в этом месяце)
#                   / (сколько было активных на начало месяца)
rows = []
for p in pd.period_range("2026-01", "2026-06", freq="M"):
    start = pd.Timestamp(f"{p}-01", tz="UTC")  # первый день месяца, полночь
    end = start + pd.offsets.MonthBegin(1) # первый день следующего месяца

    # Активна на начало месяца: началась раньше этого момента и ещё не закончилась
    # (ended_at пустой — ещё идёт; или закончилась не раньше начала месяца)
    active = (s["started_at"] < start) & (s["ended_at"].isna() | (s["ended_at"] >= start))
        # Закончилась именно в этом месяце: дата окончания внутри [start, end).
    in_month = s["ended_at"].notna() & (s["ended_at"] >= start) & (s["ended_at"] < end)
    for t in sorted(s["tariff_id"].unique()):
        mt = s["tariff_id"] == t
        cancelled = s[in_month & mt]
        rows.append({
            "month": str(p), "tariff_id": t,
            "active": int((active & mt).sum()), # знаменатель оттока
            "ended": int((active & mt & in_month).sum()), # числитель: активные, которые закончились
            "cancels": len(cancelled), #все окончания в месяце
            "too_exp": int((cancelled["cancel_reason"] == "too_expensive").sum()),
        })
tab = pd.DataFrame(rows)
tab["churn_pct"] = (100 * tab["ended"] / tab["active"].where(tab["active"] > 0)).round(2)
tab["too_exp_share_pct"] = (100 * tab["too_exp"] / tab["cancels"].where(tab["cancels"] > 0)).round(1)
print(tab.to_string(index=False))

# До и после 1 марта по тарифам: средний месячный отток 
for t, price in ((1, "+25 %"), (2, "+12,5 %")):
    b = tab[(tab["tariff_id"] == t) & (tab["month"] < "2026-03")]
    a = tab[(tab["tariff_id"] == t) & (tab["month"] >= "2026-03")]
    # Суммируем подписко-месяцы риска и окончания, чтобы получить средний месячный отток за период.
    n1, e1, n2, e2 = b["active"].sum(), b["ended"].sum(), a["active"].sum(), a["ended"].sum()
    p1, p2 = e1 / n1, e2 / n2  # отток до и после, доли
    se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    print(f"тариф {t} ({price}): отток до {100*p1:.2f} %, после {100*p2:.2f} %, "
          f"разница {100*(p2-p1):+.2f} п.п., шум ±{100*1.96*se:.2f} п.п., "
          f"отмен в месяц в среднем {tab[tab['tariff_id'] == t]['ended'].mean():.0f}")
    print(f"    минимальный заметный сдвиг ≈ {100*2.8*se:.2f} п.п.")

подписок: 56916  с датой окончания: 21253
cancel_reason
NaN               35663
not_using          7221
too_expensive      5113
switched           3883
payment_failed     2940
other              2096
пробных: 9678
  month  tariff_id  active  ended  cancels  too_exp  churn_pct  too_exp_share_pct
2026-01          1   21936    355      356       89       1.62               25.0
2026-01          2   13416    219      220       64       1.63               29.1
2026-02          1   23139    358      358       74       1.55               20.7
2026-02          2   14100    187      187       44       1.33               23.5
2026-03          1   24276    370      374       91       1.52               24.3
2026-03          2   14746    223      223       56       1.51               25.1
2026-04          1   25432    363      363       74       1.43               20.4
2026-04          2   15583    244      244       61       1.57               25.0
2026-05          1   26607    721      725      

In [34]:
# хочу дополнительно проверить из чего состоят отмены по месяцам
c = s[s["ended_at"].notna() & (s["ended_at"] >= pd.Timestamp("2026-01-01", tz="UTC"))
      & (s["ended_at"] < pd.Timestamp("2026-07-01", tz="UTC"))].copy()
c["month"] = c["ended_at"].dt.strftime("%Y-%m")
print(pd.crosstab(c["month"], c["cancel_reason"].fillna("нет"), normalize="index").mul(100).round(1).to_string())
print(c.groupby("month")["is_trial"].mean().mul(100).round(1).to_string())

cancel_reason  not_using  other  payment_failed  switched  too_expensive
month                                                                   
2026-01             31.8    9.4            12.8      19.4           26.6
2026-02             35.8    8.6            12.3      21.7           21.7
2026-03             34.5    9.4            13.9      17.6           24.6
2026-04             36.1    8.6            13.7      19.4           22.2
2026-05             35.1   11.7            14.9      15.8           22.4
2026-06             34.4    9.3            14.4      18.1           23.7
month
2026-01    17.4
2026-02    16.0
2026-03    17.1
2026-04    18.0
2026-05    16.4
2026-06    16.7


#### ✍️ Исследование Б

Вопрос исследования - проверить, есть ли основания снизить цены на тарифы из-за волны отмен, вызванной их подорожанием. Как я это сделала - для каждого месяца с января по июнь и каждого тарифа я посчитала отток: сколько подписок, активных в начале месяца, закончилось в этом же месяце. Затем сравнила средний отток до и после повышения отдельно для «Базового»(+25%) и «Расширенного» (+12,5%) тарифов (так как если причина именно в цене более сильный рост должен быть у «Базового», который подорожал сильнее).

Также посмотрела на долю отмен с причиной too_expensive и проверила, на сколько различия могут быть разбросом. По итогу после повышения средний отток/мес вырос с 1,58% до 2,40% у «Базового» и с 1,48% до 2,46% у «Расширенного». Но в марте и апреле отток почти не изменился - 1,47% у «Базового» и 1,54% у «Расширенного», а рост начался только в мае примерно до 2,7% у обоих тарифов, и продолжился в июне, когда отток составил около 3,8%. 

- Отмен по причине цены больше не стало (это too_expensive): у «Базового» её доля была 22,8% до повышения и 22,6% после, у «Расширенного» — 26,5% и 24,2%. По месяцам она в целом держалась в том же диапазоне, а в мае–июне составила 22,4% и 23,7%. Остальные причины тоже сильно не изменились, а пробные подписки во все месяцы давали около 16–18% отмен. Поэтому рост в мае и июне нельзя связать только с ценой.

Вывод: рост оттока начался через два месяца после повышения цен на тарифы, примерно одинаково у обоих тарифов и без роста доли отмен из-за высокой цены. Возвращать старые цены только на основании этих данных я бы, скорее всего, не стала. Тем более в исследовании А я получила около 2,1 млн ₽ доп. выручки в месяц от этого повышения. 
Хотела бы также подсветить ограничения:  мой расчет не доказывает, что цена совсем не повлияла на отмены, так как, например, у меня нет данных о том, каким был бы отток без повышения. Кроме того, на основе моего опыта, я предполагаю, что новая цена начинает действовать при следующем списании примерно через 30 дней. А значит теоретически увеличение цены подписки может показывать отток через какое-то время (не сразу на следующий месяц) 
Спасибо большое! Надеюсь, Вам было интересно!